# NB5 · Stima dei volumi quando i dati sono pochissimi

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/battles5/conad-statistical-learning/blob/main/notebooks/NB5_caso_fontanelli.ipynb)

**il caso fontanelli: 3 negozi misurati, 500 da stimare, e perché qui il machine learning non basta**

corso *Apprendimento statistico* per CONAD Nord Ovest · Orso Peruzzi (IFAB)

> come si esegue una cella: clicca dentro e premi **Shift + Invio**; esegui le celle in ordine, dall'alto verso il basso.

un'insegna vuole installare **fontanelli** (erogatori d'acqua) in 500 negozi e stimare quanta acqua erogheranno. ma ne ha installati solo in **3**. la domanda: quanti litri a settimana aspettarsi sull'intera rete?

è il caso limite del mattino: **pochissime osservazioni** ($n = 3$) e più variabili che dati. vediamo cosa si può e cosa non si può fare.

> nota didattica: il dataset è sintetico e contiene di nascosto la verità per **tutti** i negozi, così alla fine possiamo svelare il totale vero e giudicare ogni approccio. nella realtà l'azienda ha solo i 3 pilota.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def carica(nome):
    "carica un csv dal repo GitHub (Colab) o dalla cartella locale ../data (fallback)"
    base = "https://raw.githubusercontent.com/battles5/conad-statistical-learning/main/data/"
    try:
        return pd.read_csv(base + nome)
    except Exception:
        return pd.read_csv("../data/" + nome)

## 1. quello che sappiamo: 3 negozi

`misurato = 1` sono i 3 negozi con il fontanello già installato (gli unici dati veri).
le variabili: tipologia, afflusso giornaliero, consumi di acqua in bottiglia, clima della zona.

In [ ]:
df = carica("fontanelli_negozi.csv")
pilota = df[df["misurato"] == 1].copy()
da_stimare = df[df["misurato"] == 0].copy()
print("negozi totali:", len(df), " | misurati:", len(pilota), " | da stimare:", len(da_stimare))

# il tasso per visita dei 3 pilota: quanto varia?
pilota["litri_per_visita"] = pilota["litri_erogati_sett"] / (pilota["visite_giorno"] * 7)
pilota[["negozio_id", "tipologia", "visite_giorno", "temp_media_zona",
        "litri_erogati_sett", "litri_per_visita"]].round(4)

già qui un campanello d'allarme: il **tasso per visita** dei 3 negozi va da circa 0,014 a 0,045, cioè varia di **3 volte**. con soli 3 punti non sappiamo quale sia quello "giusto".

## 2. la tentazione: buttiamoci un modello

proviamo comunque ad addestrare una regressione lineare sui 3 negozi, con tutte le variabili.

> **indovina prima di eseguire**: che $R^2$ farà sul training con 3 punti e 4 variabili?

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# stesse colonne per tutti, così i modelli sono allineati
Xall = pd.get_dummies(
    df[["visite_giorno", "acqua_bottiglia_litri_sett", "temp_media_zona", "tipologia"]],
    columns=["tipologia"], drop_first=True,
)
Xp = Xall[df["misurato"] == 1]
Xt = Xall[df["misurato"] == 0]
yp = df.loc[df["misurato"] == 1, "litri_erogati_sett"]

modello = LinearRegression().fit(Xp, yp)
print("R2 sul training (3 negozi):", round(r2_score(yp, modello.predict(Xp)), 3))

pred = modello.predict(Xt)
print("previsioni sui 500 negozi:")
print(f"  minima: {pred.min():.0f} litri/sett   massima: {pred.max():.0f} litri/sett")
print(f"  quante previsioni NEGATIVE (impossibili): {(pred < 0).sum()} su {len(pred)}")
print(f"  totale 'stimato' dal modello: {pred.sum():.0f} litri/sett")

$R^2 = 1{,}000$: il modello passa **perfetto** per i 3 punti. ma è una trappola: con 4 variabili e 3 dati ci sono più parametri che osservazioni, il sistema è sottodeterminato. infatti sui 500 negozi sputa previsioni assurde, perfino **negative** (litri sotto zero). è overfitting allo stato puro: il modello ha "memorizzato" i 3 negozi e non sa generalizzare.

## 3. la stima onesta: un tasso, con incertezza

niente modello: stimiamo un **tasso** dai 3 pilota e lo applichiamo ai 500, dichiarando l'incertezza.

> **manopola**: cambia `DRIVER` tra `"visite_giorno"` e `"acqua_bottiglia_litri_sett"` e guarda quale dà una banda meno larga.

In [ ]:
# >>> MANOPOLA: la variabile su cui basare il tasso <<<
DRIVER = "visite_giorno"

fattore = 7 if DRIVER == "visite_giorno" else 1   # visite/giorno -> visite/settimana
base_p = pilota[DRIVER].values * fattore
k = pilota["litri_erogati_sett"].values / base_p
base_t = da_stimare[DRIVER].values * fattore

stima_punto = (base_t * k.mean()).sum()
stima_bassa = (base_t * k.min()).sum()
stima_alta = (base_t * k.max()).sum()

print(f"driver: {DRIVER}")
print(f"tassi dei 3 pilota: {np.round(k, 4)}")
print(f"stima dei 500 negozi (tasso medio): {stima_punto:,.0f} litri/sett")
print(f"banda di incertezza (min-max tasso): {stima_bassa:,.0f}  -  {stima_alta:,.0f} litri/sett")

la stima puntuale dà un ordine di grandezza, ma la **banda è enorme**: con 3 negozi non possiamo essere precisi, e dirlo è parte del lavoro. la decisione (quanti fontanelli, che capienza) va presa tenendo conto di tutta la banda, non del solo numero centrale.

## 4. e se misurassimo più negozi?

la vera soluzione non è statistica, è raccogliere dati: simuliamo un **pilota da 30 negozi** (li installiamo e misuriamo per qualche settimana). ora una semplice regressione funziona.

In [ ]:
from sklearn.model_selection import train_test_split

noti = da_stimare.sample(30, random_state=1)          # i 30 che misuriamo
resto = da_stimare.drop(noti.index)                    # gli altri 470, da prevedere
Xn = Xall.loc[noti.index]; yn = noti["litri_erogati_sett"]

# stima onesta dell'errore con un train/test interno ai 30
Xtr, Xte, ytr, yte = train_test_split(Xn, yn, test_size=10, random_state=0)
reg = LinearRegression().fit(Xtr, ytr)
print("R2 su negozi nuovi (test interno ai 30):", round(r2_score(yte, reg.predict(Xte)), 3))

# rifit su tutti i 30 e stima del totale dei 500 = 30 misurati + 470 previsti
reg_full = LinearRegression().fit(Xn, yn)
pred_resto = np.clip(reg_full.predict(Xall.loc[resto.index]), 0, None)
stima_reg = yn.sum() + pred_resto.sum()
print(f"stima dei 500 negozi (pilota da 30): {stima_reg:,.0f} litri/sett")

con 30 negozi ben scelti l'$R^2$ sui negozi nuovi diventa decente e la stima del totale si stringe: ora **c'è un vero train/test**, e i 470 errori di previsione in parte si compensano nella somma.

## 5. il verdetto

sveliamo la verità nascosta nel dataset e confrontiamo i tre approcci sul totale dei 500 negozi.

In [ ]:
from sklearn.metrics import mean_absolute_error

vero = da_stimare["litri_erogati_sett"].values
vero_totale = vero.sum()

# stima per ogni negozio, coi tre approcci
stime = {
    "modello sui 3 negozi": pred,
    "tasso (stima puntuale)": base_t * k.mean(),
    "regressione su 30 negozi": np.clip(reg_full.predict(Xt), 0, None),
}

righe = []
for nome, s in stime.items():
    righe.append({
        "metodo": nome,
        "totale_litri_sett": round(float(np.sum(s))),
        "errore_totale_%": round(100 * (np.sum(s) - vero_totale) / vero_totale, 1),
        "errore_per_negozio_litri": round(mean_absolute_error(vero, s)),
    })
riassunto = pd.DataFrame(righe)

print(f"TOTALE VERO dei 500 negozi: {vero_totale:,.0f} litri/sett\n")
print(riassunto.to_string(index=False))
print(f"\n(il modello sui 3 negozi produce anche {int((pred < 0).sum())} previsioni negative, impossibili)")


attenzione a non farsi ingannare dal **totale**: il modello sui 3 negozi ci azzecca quasi, ma per puro caso, perché gli errori enormi sui singoli negozi si compensano nella somma. l'**errore per negozio**, invece, lo smaschera: ed è quello che conta, perché capienza e priorità si decidono negozio per negozio.

leggendo la colonna giusta:

- il **modello sui 3 negozi** è inservibile: errore per negozio altissimo e previsioni perfino negative, anche se il totale cade vicino per fortuna;
- il **tasso** dà un ordine di grandezza ragionevole con incertezza dichiarata: una buona prima stima;
- la **regressione su 30 negozi** è la migliore sia sul totale sia per negozio: la differenza non è il modello, sono i **dati**.

morale, che richiama il mattino: con $n$ piccolissimo la varianza domina e nessun modello flessibile aiuta; servono ipotesi forti più incertezza dichiarata, oppure un **pilota più ampio**. è anche la prima cosa da valutare quando arriva un progetto di data science: *ci sono abbastanza dati per imparare?*


In [ ]:
# BONUS
litri = pilota["litri_erogati_sett"].values
visite_sett = pilota["visite_giorno"].values * 7
for i in range(3):
    altri = [j for j in range(3) if j != i]
    k_due = (litri[altri] / visite_sett[altri]).mean()      # tasso dai due "noti"
    prev = visite_sett[i] * k_due                            # previsione sul terzo
    vero = litri[i]
    print(f"negozio escluso {pilota['negozio_id'].iloc[i]}: previsto {prev:.0f}, vero {vero:.0f}, "
          f"errore {100*(prev-vero)/vero:+.0f}%")